# Tensor Math Filling Demo (Candle)

This notebook showcases generating tensors from analytical mathematical expressions using Candle.
We build coordinate grids and evaluate formulas like: 
- $f(x,y) = in(a x) os(b y)$
- $g(x,y) = e^{-((x-x_0)^2 + (y-y_0)^2)/(2\sigma^2)}$ (Gaussian bump)
- Radial distance: $r(x,y) = qrt{(x-x_c)^2 + (y-y_c)^2}$ (normalized)
- Checkerboard: $c(x,y) = ((floor x / s 
floor + floor y / s 
floor) \bmod 2)$

We'll also parse free-form expressions (future: via `arithmetic_parser`) to turn strings into tensor values.

In [5]:
// Notebook environment setup
// If using the evcxr Rust kernel, bring in dependencies. Adjust paths if kernel runs outside workspace root.
:dep candle-core = { path = "../../candle-core", default-features = false }
:dep candle-notebooks = { path = "." }
:dep anyhow = "1"
// (If path resolution fails, replace path with a versioned crate or absolute path.)

use candle_core::{Tensor, DType, Device};
use anyhow::Result;
use std::f64::consts::PI;

// Helper: print build/version info
fn show_env(device: &Device) {
    println!("Device: {:?}", device);
    #[cfg(debug_assertions)]
    println!("Build: debug");
    #[cfg(not(debug_assertions))]
    println!("Build: release");
}

let device = Device::Cpu;
show_env(&device);

Device: Cpu
Build: debug
Build: debug


In [ ]:
// Set cwd to repo root for consistent relative paths.
candle_notebooks::set_notebook_cwd().unwrap();
println!("CWD set to repo root: {:?}", std::env::current_dir().unwrap());

// Configure image store output directory for saved PNGs.
use candle_notebooks as nb;
nb::set_image_store_rel_dir("images_store").unwrap();
println!("Image store directory: images_store");

## Coordinate Grid Helper
We create a function to build 2D coordinate grids X, Y over ranges with optional centering.

In [8]:
fn make_xy(nx: usize, ny: usize, x_min: f64, x_max: f64, y_min: f64, y_max: f64, device: &Device) -> Result<(Tensor, Tensor)> {
    let xs: Vec<f64> = (0..nx).map(|i| x_min + (x_max - x_min) * (i as f64) / (nx as f64 - 1.0)).collect();
    let ys: Vec<f64> = (0..ny).map(|j| y_min + (y_max - y_min) * (j as f64) / (ny as f64 - 1.0)).collect();
    let x = Tensor::new(xs, device)?.reshape((nx, 1))?;
    let y = Tensor::new(ys, device)?.reshape((1, ny))?;
    // broadcast to (nx, ny)
    let X = x.broadcast_as((nx, ny))?;
    let Y = y.broadcast_as((nx, ny))?;
    Ok((X, Y))
}
let (nx, ny) = (128, 128);
let (X, Y) = make_xy(nx, ny, -1.0, 1.0, -1.0, 1.0, &device)?;
println!("Grid shape: {:?}", X.shape());

Grid shape: [128, 128]


## Analytical Functions
We'll build sample tensors using direct Rust math with Candle broadcasting.

In [14]:
// sin-cos pattern f(x,y) = sin(a x) * cos(b y)
let a = 6.0f64;
let b = 4.0f64;
let xa = (X.clone() * a)?;
let yb = (Y.clone() * b)?;
let f = (xa.sin()? * yb.cos()?)?;
let f_flat = f.flatten_all()?;
let f_min = f_flat.min(0)?.to_scalar::<f64>()?;
let f_max = f_flat.max(0)?.to_scalar::<f64>()?;
println!("f stats -> min: {:.3} max: {:.3}", f_min, f_max);

// Gaussian bump centered at (0,0) with sigma=0.35
let sigma = 0.35f64;
let x2 = (X.clone() * X.clone())?;
let y2 = (Y.clone() * Y.clone())?;
let r2 = (x2 + y2)?;
let sigma_sq = 2.0 * sigma * sigma;
let neg_exp_arg = (r2.clone() / sigma_sq)?;
let g = (neg_exp_arg * -1.0)?.exp()?;
let g_flat = g.flatten_all()?;
let g_min = g_flat.min(0)?.to_scalar::<f64>()?;
let g_max = g_flat.max(0)?.to_scalar::<f64>()?;
println!("g stats -> min: {:.3} max: {:.3}", g_min, g_max);

// Radial distance normalized
let r = r2.sqrt()?;
let r_flat = r.clone().flatten_all()?;
let r_max_val = r_flat.max(0)?.to_scalar::<f64>()?;
let r_norm = (r / r_max_val)?;

// Simple checkerboard approximation
let s = 0.15;
let x_scaled = (X.clone() / s)?.floor()?;
let y_scaled = (Y.clone() / s)?.floor()?;
let sum_scaled = (x_scaled + y_scaled)?;

println!("Analytical functions computed successfully");

f stats -> min: -1.000 max: 1.000
g stats -> min: 0.000 max: 0.999
Analytical functions computed successfully
g stats -> min: 0.000 max: 0.999
Analytical functions computed successfully


## Normalization Helper for Image Saving
We map a tensor to 0..255 u8 for quick visualization (future: integrate with image store).

In [16]:
fn to_u8(t: &Tensor) -> Result<Vec<u8>> {
    let t_flat = t.flatten_all()?;
    let min = t_flat.min(0)?.to_scalar::<f64>()?;
    let max = t_flat.max(0)?.to_scalar::<f64>()?;
    let span = max - min;
    
    // Avoid div-by-zero: if span==0 just zeros
    let norm = if span == 0.0 {
        (t - min)?
    } else {
        ((t - min)? / span)?
    };
    let scaled = (norm * 255.0)?;
    let scaled_vec = scaled.flatten_all()?.to_vec1::<f64>()?;
    Ok(scaled_vec.into_iter().map(|v| v.round().clamp(0.0, 255.0) as u8).collect())
}

let f_u8 = to_u8(&f)?;
println!("u8 buffer length (f): {}", f_u8.len());

u8 buffer length (f): 16384


## (Planned) Expression Parsing
Future step: integrate `arithmetic_parser` so a user string like `sin(6*x)*cos(4*y)` is parsed and evaluated over X,Y.
We would:
1. Create a context binding symbols x,y,pi.
2. Parse expression to an AST.
3. Evaluate per element (or vectorize by mapping over flattened arrays, then reshaping).
Once stable, wrap in a helper `eval_expr(expr: &str, bindings: &HashMap<&str,&Tensor>)`.

## Next Ideas
- Integrate image saving to `images_store/`.
- Add dynamic parameter sliders (in egui context) for a, b, sigma.
- Add 3D stack: vary t and evaluate f_t(x,y) = sin(a x + t)*cos(b y - t).
- Inline LaTeX rendering for formulas to keep code + math in sync.

## Parsed Expression Integration (B3)

Below we demonstrate the new expression parser (`ExprEnv`, `eval_expr`) on the same grids used earlier.

Goals:
- Define grids X, Y (already created earlier) and parameter map.
- Evaluate a list of expression strings.
- Compare each parsed result to a manual construction when applicable.
- Show max absolute difference.
- Gaussian example using parameters (a, b, sigma).
- Timing snapshot.

We'll re-create X, Y if not in scope (idempotent).

In [18]:
// Recreate grid (idempotent) and import parser
use std::collections::HashMap;
use candle_core::{Tensor, Device, DType};
use candle_notebooks::expr::{ExprEnv, eval_expr};

let dev = Device::Cpu;
let n = 128usize;
let xs: Vec<f32> = (0..n).map(|i| (i as f32)/(n as f32 - 1.0) * 2.0 - 1.0).collect();
let ys = xs.clone();
let x = Tensor::from_vec(xs.clone(), (n,), &dev)?.to_dtype(DType::F32)?;
let y = Tensor::from_vec(ys.clone(), (n,), &dev)?.to_dtype(DType::F32)?;
let xx = x.unsqueeze(0)?.broadcast_as((n, n))?; // row-wise
let yy = y.unsqueeze(1)?.broadcast_as((n, n))?; // column-wise

let mut params = HashMap::new();
params.insert("a".to_string(), 1.0f64);
params.insert("b".to_string(), 1.0f64);
params.insert("sigma".to_string(), 0.35f64);

let env = ExprEnv::new(xx.clone().to_dtype(DType::F64)?, yy.clone().to_dtype(DType::F64)?, params)?;
println!("Grid & env ready: shape={:?}", env.x.shape());

Grid & env ready: shape=[128, 128]


In [20]:
// Helper to evaluate an expression and optionally compare against a manual tensor builder closure.
use std::time::Instant;
use candle_core::{Tensor, DType};
use candle_notebooks::expr::{ExprEnv, eval_expr};

fn eval_and_compare<F>(label: &str, expr: &str, env: &ExprEnv, manual: Option<F>) -> anyhow::Result<()>
where F: Fn() -> candle_core::Result<Tensor>
{
    let t0 = Instant::now();
    let parsed = eval_expr(expr, env)?; // returns f32 tensor
    let dt = t0.elapsed();
    if let Some(build) = manual {
        let manual_t = build()?; // assume already f32 shape match
        let diff = (&parsed - &manual_t)?.abs()?.flatten_all()?.max(0)?;
        println!("{label}: expr='{}' max_abs_diff={:.3e} time_ms={:.2}", expr, diff.to_scalar::<f32>()?, dt.as_secs_f64()*1e3);
    } else {
        println!("{label}: expr='{}' time_ms={:.2}", expr, dt.as_secs_f64()*1e3);
    }
    Ok(())
}

// Prepare some manual comparison builders (capturing xx, yy)
let xx_f32 = env.x.to_dtype(DType::F32)?;
let yy_f32 = env.y.to_dtype(DType::F32)?;

// Run a few baseline expressions

eval_and_compare("radial", "sqrt(x*x + y*y)", &env, Some(|| {
    let xx_sq = (&xx_f32 * &xx_f32)?;
    let yy_sq = (&yy_f32 * &yy_f32)?;
    let sum = (xx_sq + yy_sq)?;
    Ok(sum.sqrt()?)
}))?;

eval_and_compare("checker", "(sin(10*x)*sin(10*y))", &env, Some(|| {
    let x_10 = (&xx_f32 * 10.0)?;
    let y_10 = (&yy_f32 * 10.0)?;
    Ok((x_10.sin()? * y_10.sin()?)?)
}))?;

eval_and_compare("gaussian_basic", "exp(-(x*x + y*y))", &env, None::<fn() -> candle_core::Result<Tensor>>)?;

radial: expr='sqrt(x*x + y*y)' max_abs_diff=1.192e-7 time_ms=0.22
checker: expr='(sin(10*x)*sin(10*y))' max_abs_diff=3.874e-7 time_ms=0.46
gaussian_basic: expr='exp(-(x*x + y*y))' time_ms=0.29
checker: expr='(sin(10*x)*sin(10*y))' max_abs_diff=3.874e-7 time_ms=0.46
gaussian_basic: expr='exp(-(x*x + y*y))' time_ms=0.29


In [22]:
// Gaussian with parameters a, b, sigma in params
use candle_notebooks::expr::ExprEnv; // ensure visibility (noop if already imported earlier)

eval_and_compare(
    "gaussian_param",
    "a * exp(-((x*x)/(2*sigma*sigma) + (y*y)/(2*sigma*sigma))) * b",
    &env,
    None::<fn() -> candle_core::Result<Tensor>>,
)?;

// More complex combo expression

eval_and_compare(
    "combo",
    "sin(3*x) * cos(5*y) + 0.25 * exp(-(x*x + y*y))",
    &env,
    None::<fn() -> candle_core::Result<Tensor>>,
)?;

println!("Done B3 expression evaluations.");

gaussian_param: expr='a * exp(-((x*x)/(2*sigma*sigma) + (y*y)/(2*sigma*sigma))) * b' time_ms=0.70
combo: expr='sin(3*x) * cos(5*y) + 0.25 * exp(-(x*x + y*y))' time_ms=0.66
Done B3 expression evaluations.
combo: expr='sin(3*x) * cos(5*y) + 0.25 * exp(-(x*x + y*y))' time_ms=0.66
Done B3 expression evaluations.


In [25]:
// Validation: re-evaluate a few expressions and assert numeric closeness.
use candle_core::{DType, Tensor};
use candle_notebooks::expr::{ExprEnv, eval_expr};

let tol = 1e-4f32;

// Reuse env defined earlier (radial, checker, gaussian basic)
let radial_p = eval_expr("sqrt(x*x + y*y)", &env)?;
let manual_x2 = (env.x.to_dtype(DType::F32)? * env.x.to_dtype(DType::F32)?)?;
let manual_y2 = (env.y.to_dtype(DType::F32)? * env.y.to_dtype(DType::F32)?)?;
let manual_radial = (manual_x2.clone() + manual_y2.clone())?.sqrt()?;
let diff_r = (&radial_p - &manual_radial)?.abs()?.flatten_all()?.max(0)?.to_scalar::<f32>()?;
assert!(diff_r <= tol, "radial diff {} > {}", diff_r, tol);

let checker_p = eval_expr("(sin(10*x)*sin(10*y))", &env)?;
let x_10 = (env.x.to_dtype(DType::F32)? * 10.0)?;
let y_10 = (env.y.to_dtype(DType::F32)? * 10.0)?;
let checker_m = (x_10.sin()? * y_10.sin()?)?;
let diff_c = (&checker_p - &checker_m)?.abs()?.flatten_all()?.max(0)?.to_scalar::<f32>()?;
assert!(diff_c <= tol, "checker diff {} > {}", diff_c, tol);

let gauss_p = eval_expr("exp(-(x*x + y*y))", &env)?;
// manual gaussian basic
let g_manual_sum = (manual_x2 + manual_y2)?;
let g_manual = (g_manual_sum * -1.0)?.exp()?; // exp(-r2)
let diff_g = (&gauss_p - &g_manual)?.abs()?.flatten_all()?.max(0)?.to_scalar::<f32>()?;
assert!(diff_g <= tol, "gaussian diff {} > {}", diff_g, tol);

println!("Validation passed: radial={:.2e} checker={:.2e} gaussian={:.2e}", diff_r, diff_c, diff_g);

Validation passed: radial=1.19e-7 checker=3.87e-7 gaussian=5.96e-8
